# Notebook 13 — Object Detection Basics: Single-Object Detector

## Learning Objectives
- Understand the joint classification + box regression formulation for object detection.
- Build and train `TinySingleObjectDetector` on synthetic shapes.
- Understand normalized bounding box coordinates (cx, cy, w, h) in [0, 1].
- Visualise predicted vs ground-truth bounding boxes on images.
- Evaluate with classification accuracy and mean IoU.
- Understand the role of `lambda_box` in balancing the combined loss.

## Big Picture
Object detection answers two questions simultaneously: *What* is in the image (classification)
and *Where* is it (bounding box regression). A single-object detector is the simplest case:
one class label + one box per image. This notebook introduces the core ideas before scaling
to multi-object detection in Notebooks 14 and 15.

## Dataset
**SyntheticShapes (single_detection mode)** — 128×128 RGB images, exactly one shape per image.
3 classes: 0=circle, 1=square, 2=triangle (no background class in detection).
Box format: normalised (cx, cy, w, h) in [0, 1].

## Architecture
```
Input [B, 3, 128, 128]
  └─ CNN Backbone:
     Conv(3→32)+BN+ReLU+MaxPool → [B, 32, 64, 64]
     Conv(32→64)+BN+ReLU+MaxPool → [B, 64, 32, 32]
     Conv(64→128)+BN+ReLU       → [B, 128, 32, 32]
  └─ AdaptiveAvgPool2d(1)       → [B, 128, 1, 1]
  └─ Flatten                    → [B, 128]
  ├─ Classification Head: Linear(128→64)→ReLU→Linear(64→3) → class_logits [B, 3]
  └─ Box Head:           Linear(128→64)→ReLU→Linear(64→4)→Sigmoid → boxes [B, 4]
```

## Theory
**Bounding box parametrisation**: normalised (cx, cy, w, h) where all values ∈ [0, 1].
- `cx, cy` = centre x, y relative to image width/height.
- `w, h` = width, height relative to image size.
- A sigmoid activation in the box head ensures outputs stay in [0, 1].

**Combined loss**:
$$\mathcal{L} = \underbrace{\mathcal{L}_{CE}(\text{class logits}, y)}_\text{classification} + \lambda_{box} \cdot \underbrace{\text{SmoothL1}(\hat{b}, b)}_\text{box regression}$$

**Smooth L1 loss** (Huber loss): less sensitive to outliers than MSE, more stable than L1.

**Intersection over Union (IoU)**:
$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$
IoU=0: no overlap, IoU=1: perfect match. Threshold 0.5 is commonly used for 'correct detection'.

## Implementation from Scratch
### 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root when running from notebooks/

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, update_runs_summary
from src.utils.paths import RUNS_DETECTION_DIR
from src.data.synthetic_shapes import load_shapes_data
from src.models.tiny_detector import TinySingleObjectDetector
from src.losses.detection_losses import single_object_detection_loss
from src.metrics.detection import box_cxcywh_to_xyxy, box_iou
from src.visualization.detection import draw_bounding_boxes, plot_detection_examples

seed_everything(42)
device = get_best_device()
run_dir = prepare_run_dir(RUNS_DETECTION_DIR, clean=False)
print(f'Device  : {device}')
print(f'Run dir : {run_dir}')

## Dataset
### 2. Load Synthetic Shapes (single_detection mode)

In [ ]:
IMAGE_SIZE  = 128
NUM_CLASSES = 3   # 0=circle, 1=square, 2=triangle
BATCH_SIZE  = 32
NUM_EPOCHS  = 8
LR          = 1e-3
LAMBDA_BOX  = 5.0
CLASS_NAMES = ['circle', 'square', 'triangle']

train_loader, val_loader = load_shapes_data(
    n_train=800,
    n_val=200,
    image_size=IMAGE_SIZE,
    mode='single_detection',
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

imgs, class_ids, boxes = next(iter(train_loader))
print(f'Image batch shape  : {imgs.shape}        (B, 3, H, W)')
print(f'Class IDs shape    : {class_ids.shape}   (B,)  — {torch.unique(class_ids).tolist()}')
print(f'Boxes shape        : {boxes.shape}    (B, 4) in normalised (cx,cy,w,h)')
print(f'Example box        : cx={boxes[0,0]:.3f} cy={boxes[0,1]:.3f} w={boxes[0,2]:.3f} h={boxes[0,3]:.3f}')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

In [ ]:
# Visualise sample images with ground-truth boxes
imgs_np = imgs[:4].numpy()  # [4, 3, 128, 128]

# Convert normalised cxcywh to pixel xyxy for drawing
boxes_xyxy_px = box_cxcywh_to_xyxy(boxes[:4]) * IMAGE_SIZE  # scale to pixels

plot_detection_examples(
    images=imgs_np,
    gt_boxes_list=[boxes_xyxy_px[i].numpy() for i in range(4)],
    gt_class_ids_list=[[class_ids[i].item()] for i in range(4)],
    save_path=run_dir / 'single_detector_gt_samples.png',
    title='Sample Images with Ground-Truth Boxes (single_detection mode)',
    n_examples=4,
    class_names=CLASS_NAMES,
)
print('Ground-truth samples visualised.')

### 3. Build TinySingleObjectDetector

In [ ]:
model = TinySingleObjectDetector(
    in_channels=3,
    num_classes=NUM_CLASSES,
    image_size=IMAGE_SIZE,
).to(device)

n_params = model.count_parameters()
print(f'Trainable parameters: {n_params:,}')

# Shape trace
x_sample = imgs[:4].to(device)                    # [4, 3, 128, 128]
with torch.no_grad():
    cls_logits, pred_boxes = model(x_sample)

print(f'Input shape           : {x_sample.shape}   (batch, 3, H, W)')
print(f'class_logits shape    : {cls_logits.shape}  (batch, num_classes)')
print(f'pred_boxes shape      : {pred_boxes.shape}  (batch, 4) in normalised cxcywh')
print(f'Box value range       : [{pred_boxes.min():.3f}, {pred_boxes.max():.3f}]  (sigmoid → [0,1])')

## Sanity Checks

In [ ]:
# 1. Output shapes
assert cls_logits.shape == (4, NUM_CLASSES)
assert pred_boxes.shape == (4, 4)
print('Output shape checks passed.')

# 2. Boxes in [0, 1]
assert pred_boxes.min() >= 0 and pred_boxes.max() <= 1
print('Boxes constrained to [0, 1] — check!')

# 3. No NaN
assert not torch.isnan(cls_logits).any()
assert not torch.isnan(pred_boxes).any()
print('No NaN in output.')

# 4. Loss computable
test_loss = single_object_detection_loss(
    cls_logits, pred_boxes,
    class_ids[:4].to(device), boxes[:4].to(device),
    lambda_box=LAMBDA_BOX,
)
print(f'Initial combined loss : {test_loss.item():.4f}')

# 5. IoU with random boxes ~ 0
pred_xyxy = box_cxcywh_to_xyxy(pred_boxes)
gt_xyxy   = box_cxcywh_to_xyxy(boxes[:4].to(device))
diag_ious = torch.diagonal(box_iou(pred_xyxy, gt_xyxy))
print(f'Random-init mean IoU : {diag_ious.mean().item():.4f}  (expected low)')

print('All sanity checks passed!')

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)

history = {'train_loss': [], 'val_loss': [], 'cls_accuracy': [], 'mean_iou': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, n_batches = 0.0, 0
    for batch_imgs, batch_cls, batch_boxes in train_loader:
        batch_imgs  = batch_imgs.to(device)                  # [B, 3, H, W]
        batch_cls   = batch_cls.to(device)                   # [B]
        batch_boxes = batch_boxes.to(device)                 # [B, 4]
        optimizer.zero_grad()
        cls_logits, pred_boxes = model(batch_imgs)           # [B,3], [B,4]
        loss = single_object_detection_loss(
            cls_logits, pred_boxes, batch_cls, batch_boxes, lambda_box=LAMBDA_BOX
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
        n_batches += 1
    history['train_loss'].append(t_loss / n_batches)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_acc, v_iou, v_n = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for batch_imgs, batch_cls, batch_boxes in val_loader:
            batch_imgs  = batch_imgs.to(device)
            batch_cls   = batch_cls.to(device)
            batch_boxes = batch_boxes.to(device)
            cls_logits, pred_boxes = model(batch_imgs)
            v_loss += single_object_detection_loss(
                cls_logits, pred_boxes, batch_cls, batch_boxes, LAMBDA_BOX
            ).item()
            # Accuracy
            v_acc += (cls_logits.argmax(-1) == batch_cls).float().mean().item()
            # IoU
            pred_xyxy = box_cxcywh_to_xyxy(pred_boxes)
            gt_xyxy   = box_cxcywh_to_xyxy(batch_boxes)
            iou_vals  = torch.diagonal(box_iou(pred_xyxy, gt_xyxy))
            v_iou += iou_vals.mean().item()
            v_n += 1
    history['val_loss'].append(v_loss / v_n)
    history['cls_accuracy'].append(v_acc / v_n)
    history['mean_iou'].append(v_iou / v_n)
    scheduler.step()

    print(f'Epoch {epoch}/{NUM_EPOCHS}  '
          f'train={history["train_loss"][-1]:.4f}  '
          f'val={history["val_loss"][-1]:.4f}  '
          f'cls_acc={history["cls_accuracy"][-1]:.4f}  '
          f'mIoU={history["mean_iou"][-1]:.4f}')

print('Training complete!')

## Evaluation

In [ ]:
model.eval()
all_preds_cls, all_gt_cls = [], []
all_pred_xyxy, all_gt_xyxy = [], []

with torch.no_grad():
    for batch_imgs, batch_cls, batch_boxes in val_loader:
        batch_imgs  = batch_imgs.to(device)
        cls_logits, pred_boxes = model(batch_imgs)
        all_preds_cls.extend(cls_logits.argmax(-1).cpu().tolist())
        all_gt_cls.extend(batch_cls.tolist())
        all_pred_xyxy.append(box_cxcywh_to_xyxy(pred_boxes).cpu())
        all_gt_xyxy.append(box_cxcywh_to_xyxy(batch_boxes))

all_pred_xyxy = torch.cat(all_pred_xyxy)
all_gt_xyxy   = torch.cat(all_gt_xyxy)
ious = torch.diagonal(box_iou(all_pred_xyxy, all_gt_xyxy))

final_acc  = sum(p == g for p, g in zip(all_preds_cls, all_gt_cls)) / len(all_gt_cls)
final_miou = ious.mean().item()
iou50_rate = (ious >= 0.5).float().mean().item()  # fraction with IoU >= 0.5

print(f'Classification accuracy : {final_acc:.4f}  ({final_acc*100:.1f}%)')
print(f'Mean IoU               : {final_miou:.4f}')
print(f'IoU@0.5 rate           : {iou50_rate:.4f}  (fraction of boxes with IoU >= 0.5)')

metrics = {
    'cls_accuracy': float(final_acc),
    'mean_iou': float(final_miou),
    'iou_at_50': float(iou50_rate),
    'final_train_loss': float(history['train_loss'][-1]),
    'final_val_loss': float(history['val_loss'][-1]),
    'num_params': n_params,
    'num_epochs': NUM_EPOCHS,
    'lambda_box': LAMBDA_BOX,
}
save_metrics_json(metrics, run_dir / 'single_detector_metrics.json')
print(f'Metrics saved.')

## Visualization

In [ ]:
# Training curves
from src.visualization.plots import plot_training_curves
plot_training_curves(
    {'train_loss': history['train_loss'], 'val_loss': history['val_loss'],
     'cls_accuracy': history['cls_accuracy'], 'mean_iou': history['mean_iou']},
    save_path=run_dir / 'single_detector_training_curve.png',
    title='TinySingleObjectDetector Training Curves',
)
print('Training curve saved.')

In [ ]:
# Predicted vs ground-truth boxes
model.eval()
vis_imgs, vis_cls, vis_boxes = next(iter(val_loader))
with torch.no_grad():
    vis_cls_logits, vis_pred_boxes = model(vis_imgs.to(device))

n_vis = 4
pred_xyxy_px = (box_cxcywh_to_xyxy(vis_pred_boxes[:n_vis].cpu()) * IMAGE_SIZE).numpy()
gt_xyxy_px   = (box_cxcywh_to_xyxy(vis_boxes[:n_vis]) * IMAGE_SIZE).numpy()
pred_cls_ids = vis_cls_logits[:n_vis].argmax(-1).cpu().tolist()
gt_cls_ids   = vis_cls[:n_vis].tolist()
pred_scores  = vis_cls_logits[:n_vis].softmax(-1).max(-1).values.cpu().tolist()

plot_detection_examples(
    images=vis_imgs[:n_vis].numpy(),
    gt_boxes_list=[gt_xyxy_px[i:i+1] for i in range(n_vis)],
    gt_class_ids_list=[[gt_cls_ids[i]] for i in range(n_vis)],
    pred_boxes_list=[pred_xyxy_px[i:i+1] for i in range(n_vis)],
    pred_class_ids_list=[[pred_cls_ids[i]] for i in range(n_vis)],
    pred_scores_list=[[pred_scores[i]] for i in range(n_vis)],
    save_path=run_dir / 'single_detector_predictions.png',
    title='Single-Object Detector: GT (left) vs Predicted (right)',
    n_examples=n_vis,
    class_names=CLASS_NAMES,
)
print('Prediction visualisation saved.')

In [ ]:
# IoU histogram
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ious.numpy(), bins=20, range=(0, 1), color='steelblue', edgecolor='black', alpha=0.8)
ax.axvline(0.5, color='red', linestyle='--', label='IoU=0.5 threshold')
ax.axvline(ious.mean().item(), color='orange', linestyle='-', label=f'Mean IoU={ious.mean():.3f}')
ax.set_xlabel('IoU')
ax.set_ylabel('Count')
ax.set_title('Distribution of Predicted Box IoU (Validation Set)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(run_dir / 'single_detector_iou_histogram.png', dpi=120)
plt.close(fig)
print('IoU histogram saved.')

## Failure Cases

**Common failure patterns:**

1. **lambda_box too small**: The model focuses on classification and ignores box regression.
   Try `lambda_box=0.1` — boxes will be scattered randomly while accuracy stays high.

2. **Predicted boxes too small or too large**: The Sigmoid activation limits boxes to [0, 1]
   but doesn't constrain w ≥ 0.05. Small objects may get predicted with near-zero width.

3. **Classification vs regression competition**: With equal weight, the model may sacrifice
   box precision for better class prediction when they conflict.

4. **All boxes regress to the mean**: With random init, the box head may converge to predicting
   the average box (center, 0.4 width). This is a local minimum — a lower learning rate with
   warmup can help escape it.

5. **IoU near 0 for entire training**: This can happen when the Sigmoid-bounded box output is
   near 0.5 (centre, zero size). Check that box loss is actually decreasing.

## Exercises

1. **lambda_box ablation**: Train with `lambda_box` ∈ {0.1, 1.0, 5.0, 10.0}. Plot mean IoU
   vs lambda_box. Is there an optimal value?

2. **GIoU loss**: Replace SmoothL1 with Generalised IoU loss. GIoU has better gradient flow
   when boxes don't overlap. Does it improve convergence speed?

3. **IoU threshold sweep**: Compute IoU@0.25, IoU@0.5, IoU@0.75. Plot the detection rate vs
   IoU threshold curve. This is the precursor to mAP evaluation.

4. **Deeper backbone**: Replace the 3-block CNN backbone with a pre-trained ResNet-18 (frozen).
   Compare parameter count and accuracy.

5. **Calibration plot**: Plot predicted confidence (max softmax) vs actual accuracy. Is the
   model well-calibrated? (Confident predictions → high accuracy?)

## Key Takeaways

- **Object detection = classification + box regression**: two heads share the same CNN backbone.
- **Normalised (cx, cy, w, h)** coordinates are easier to learn than pixel coordinates because
  they are scale-independent and bounded in [0, 1].
- **Smooth L1 loss** for boxes: more stable than MSE for large residuals, converges faster than L1.
- **lambda_box** controls the classification-regression balance: too small → bad boxes, too large → bad class.
- **IoU@0.5** is the standard detection threshold: a predicted box is 'correct' if it overlaps
  the ground truth by at least 50%.
- Single-object detection is the simplest case. Notebooks 14 and 15 extend to multi-object detection.

## Saved Outputs

In [ ]:
save_markdown_report(
    title='Single-Object Detector — TinySingleObjectDetector',
    summary=(
        f'TinySingleObjectDetector trained {NUM_EPOCHS} epochs on SyntheticShapes (single_detection). '
        f'Classification acc: {final_acc:.4f}, Mean IoU: {final_miou:.4f}, IoU@0.5: {iou50_rate:.4f}.'
    ),
    metrics=metrics,
    figures=['single_detector_training_curve.png', 'single_detector_predictions.png',
             'single_detector_iou_histogram.png'],
    tables=[],
    output_path=run_dir / 'single_detector_report.md',
    device=str(device),
    hyperparams={
        'image_size': IMAGE_SIZE, 'num_classes': NUM_CLASSES, 'lambda_box': LAMBDA_BOX,
        'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS, 'lr': LR,
    },
    architecture='CNNBackbone(3→128) + GlobalPool → ClassHead(3) + BoxHead(4, sigmoid)',
    loss_fn='CrossEntropy + lambda_box * SmoothL1',
)

update_runs_summary(
    session_name='single_object_detection',
    report_path=run_dir / 'single_detector_report.md',
    metrics=metrics,
    figures=['single_detector_training_curve.png', 'single_detector_predictions.png'],
)

print('All outputs saved:')
print(f'  {run_dir}/single_detector_metrics.json')
print(f'  {run_dir}/single_detector_training_curve.png')
print(f'  {run_dir}/single_detector_predictions.png')
print(f'  {run_dir}/single_detector_report.md')